# SWE-RL Re-implementation: End-to-End Workflow

## 0) Environment and helper utilities

This cell sets paths, helper functions, and figure output folders.
It also provides a single command runner to keep logs readable.

In [1]:
from pathlib import Path
import subprocess
import json
import matplotlib.pyplot as plt

ROOT = Path.cwd()
FIG_DIR = ROOT / 'outputs' / 'notebook_figs'
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_CFG = 'configs/data_config_student.yaml'
TRAIN_CFG = 'configs/train_config_student.yaml'

print('Workspace:', ROOT)
print('Figure dir:', FIG_DIR)

def run_cmd(cmd: str, check: bool = True):
    print(f"\n$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed ({result.returncode}): {cmd}')
    return result


Workspace: d:\Software\claude\LLM 8803\swe_rl_reimplement
Figure dir: d:\Software\claude\LLM 8803\swe_rl_reimplement\outputs\notebook_figs


## 1) Pipeline overview

### Data stage
1. Fetch PR events from GHArchive.
2. Filter for high-quality bug-fix PR candidates.
3. Extract `(issue, code context, oracle patch)` triples.
4. Build local FAISS retrieval index.

### Training stage
5. Generate SFT data (oracle mode by default, no paid API).
6. Train SFT baseline.
7. Train GRPO policy.

### Evaluation stage
8. Run inference on SWE-bench-style dataset.
9. Build submission file and (optionally) offline validation reward.

## 1.5) Prerequisites and environment checks

Before running heavy stages, verify:
- Python environment is active in this notebook kernel.
- `pip install -r requirements.txt` has been run.
- Optional: `GITHUB_TOKEN` is set for higher GitHub API rate limits.

In [2]:
import os
import shutil

print('Python executable:', shutil.which('python'))
print('Has GITHUB_TOKEN:', bool(os.getenv('GITHUB_TOKEN')))

required = [
    Path('run.py'),
    Path(DATA_CFG),
    Path(TRAIN_CFG),
]
for p in required:
    print(f"{'OK  ' if p.exists() else 'MISS'}  {p}")

Python executable: d:\Software\Miniconda\envs\3dunet\python.EXE
Has GITHUB_TOKEN: False
OK    run.py
OK    configs\data_config_student.yaml
OK    configs\train_config_student.yaml


## 2) Quick sanity checks

Check CLI availability before long runs.

In [3]:
run_cmd('python run.py --help')
run_cmd('python run.py infer --help')


$ python run.py --help
usage: run.py [-h] {data,sft_data,sft_train,train,infer,eval} ...

SWE-RL Re-implementation Pipeline Runner

positional arguments:
  {data,sft_data,sft_train,train,infer,eval}
    data                Run data preprocessing pipeline
    sft_data            Generate SFT CoT training data
    sft_train           Train SFT baseline
    train               Run GRPO (SWE-RL) training
    infer               Run inference on SWE-bench Verified
    eval                Post-process and evaluate

options:
  -h, --help            show this help message and exit

run.py — Top-level orchestrator for SWE-RL Re-implementation
Convenience script to run any stage of the pipeline from one place.

Usage:
    # Full data pipeline
    python run.py data --config configs/data_config.yaml

    # Single data stage
    python run.py data --stage fetch --config configs/data_config.yaml
    python run.py data --stage filter --config configs/data_config.yaml
    python run.py data --stage 

CompletedProcess(args='python run.py infer --help', returncode=0, stdout='usage: run.py infer [-h] --model_path MODEL_PATH [--config CONFIG]\n                    [--output_dir OUTPUT_DIR] [--dataset DATASET]\n                    [--num_samples NUM_SAMPLES] [--temperature TEMPERATURE]\n                    [--max_new_tokens MAX_NEW_TOKENS]\n                    [--top_k_chunks TOP_K_CHUNKS]\n                    [--repo_cache_dir REPO_CACHE_DIR]\n                    [--embed_model EMBED_MODEL]\n                    [--max_eval_files MAX_EVAL_FILES]\n                    [--batch_size BATCH_SIZE]\n\noptions:\n  -h, --help            show this help message and exit\n  --model_path MODEL_PATH\n  --config CONFIG       Optional train config with evaluation section\n  --output_dir OUTPUT_DIR\n  --dataset DATASET\n  --num_samples NUM_SAMPLES\n  --temperature TEMPERATURE\n  --max_new_tokens MAX_NEW_TOKENS\n  --top_k_chunks TOP_K_CHUNKS\n  --repo_cache_dir REPO_CACHE_DIR\n  --embed_model EMBED_MODEL\

## 3) Run data pipeline

This builds `data/raw`, `data/processed`, and `data/rag` artifacts.

> For debugging, you can run one stage at a time with `--stage fetch|filter|extract|index`.

python run.py data --stage fetch --config configs/data_config.yaml

python run.py data --stage filter --config configs/data_config.yaml

python run.py data --stage extract --config configs/data_config.yaml

python run.py data --stage index --config configs/data_config.yaml

In [ ]:
run_cmd(f'python run.py data --config {DATA_CFG}')

## 4) Generate SFT data (oracle mode, free)

Uses oracle file deltas to create SEARCH/REPLACE supervision examples.
No paid model API is needed in student config.

In [ ]:
run_cmd(f'python run.py sft_data --config {TRAIN_CFG}')

## 5) Train SFT and GRPO

These are heavy steps. Keep them enabled for full pipeline runs.
Set `RUN_HEAVY = False` to skip when smoke-testing notebook structure.

In [ ]:
RUN_HEAVY = True

if RUN_HEAVY:
    run_cmd(f'python run.py sft_train --config {TRAIN_CFG}')
    run_cmd(f'python run.py train --config {TRAIN_CFG}')
else:
    print('Skipped heavy training steps.')

## 6) Run inference

Use low-memory safe defaults for first run (`--batch_size 1 --max_new_tokens 128`).
If your cluster has enough memory, increase these later.

In [ ]:
MODEL_PATH = 'outputs/grpo_student/final'
run_cmd(
    f'python run.py infer --model_path {MODEL_PATH} --config {TRAIN_CFG} --batch_size 1 --max_new_tokens 128'
)

## 7) Build submission and optional offline reward

Submission conversion turns SEARCH/REPLACE model outputs into unified diffs.
Offline reward is optional and useful for fast validation diagnostics.

In [ ]:
RAW_OUT = 'outputs/eval_student'
run_cmd(f'python run.py eval --mode submission --raw_output_dir {RAW_OUT}')

# Optional:
# run_cmd(f'python -m evaluation.evaluate --mode val_reward --raw_output_dir {RAW_OUT} --val_file data/processed/val.jsonl --reward_mode combined')

## 8) Load reports and plot key metrics

This cell reads JSON reports (if present), creates summary figures, saves them to `outputs/notebook_figs`, and displays them inline.

In [ ]:
format_report_path = ROOT / 'outputs' / 'eval_student' / 'format_report.json'
reward_report_path = ROOT / 'outputs' / 'eval_student' / 'val_reward_combined_report.json'

format_report = None
reward_report = None

if format_report_path.exists():
    format_report = json.loads(format_report_path.read_text(encoding='utf-8'))
    print('Loaded:', format_report_path)
else:
    print('Missing:', format_report_path)

if reward_report_path.exists():
    reward_report = json.loads(reward_report_path.read_text(encoding='utf-8'))
    print('Loaded:', reward_report_path)
else:
    print('Optional missing:', reward_report_path)

labels = []
values = []

if format_report is not None:
    labels.append('format_accuracy')
    values.append(float(format_report.get('format_accuracy', 0.0)))

if reward_report is not None:
    labels.extend(['mean_reward', 'mean_max_reward'])
    values.extend([
        float(reward_report.get('mean_reward_across_instances', 0.0)),
        float(reward_report.get('mean_max_reward', 0.0)),
    ])

if labels:
    plt.figure(figsize=(8, 4))
    bars = plt.bar(labels, values)
    plt.ylim(0, max(1.0, max(values) * 1.2))
    plt.title('SWE-RL Pipeline Summary Metrics')
    plt.ylabel('Value')
    for bar, v in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{v:.3f}', ha='center', va='bottom')
    fig_path = FIG_DIR / 'summary_metrics.png'
    plt.tight_layout()
    plt.savefig(fig_path, dpi=150)
    plt.show()
    print('Saved figure:', fig_path)
else:
    print('No reports found to plot yet. Run inference/eval cells first.')

## 9) Data artifact checklist

Use this checklist to verify each stage wrote outputs correctly.

## 9.5) Optional baseline runs for meaningful comparison

For rubric-ready comparison (Base vs SFT vs GRPO), run inference on:
- base model (from config `model.name_or_path`),
- SFT model (`outputs/sft_student/final`),
- GRPO model (`outputs/grpo_student/final`).

Use separate output directories for each run.

In [ ]:
# Optional baseline inference commands (uncomment to run)

# Base model example (replace with your actual base model path/name if local)
# run_cmd('python run.py infer --model_path Qwen/Qwen2.5-Coder-0.5B-Instruct --config configs/train_config_student.yaml --output_dir outputs/eval_base_student --batch_size 1 --max_new_tokens 128')

# SFT model
# run_cmd('python run.py infer --model_path outputs/sft_student/final --config configs/train_config_student.yaml --output_dir outputs/eval_sft_student --batch_size 1 --max_new_tokens 128')

# GRPO model (already run in main flow)
# run_cmd('python run.py infer --model_path outputs/grpo_student/final --config configs/train_config_student.yaml --output_dir outputs/eval_grpo_student --batch_size 1 --max_new_tokens 128')

## 10) Rubric-facing outputs (comparison + analysis scaffold)

This section helps ensure your final report has required evidence:
- meaningful comparison across Base / SFT / GRPO,
- quantitative metrics table,
- qualitative analysis notes.

Fill in available values after runs.

In [ ]:
# Editable comparison scaffold for report tables/plots
comparison = {
    'Base': {'format_accuracy': None, 'mean_reward': None, 'resolve_rate': None},
    'SFT':  {'format_accuracy': None, 'mean_reward': None, 'resolve_rate': None},
    'GRPO': {'format_accuracy': None, 'mean_reward': None, 'resolve_rate': None},
}

print('Fill this dictionary with your measured values before final plotting/reporting:')
print(json.dumps(comparison, indent=2))

# Optional quick plot for any filled metrics
metric = 'format_accuracy'  # change to mean_reward / resolve_rate
labels, vals = [], []
for k, v in comparison.items():
    m = v.get(metric)
    if m is not None:
        labels.append(k)
        vals.append(float(m))

if labels:
    plt.figure(figsize=(6, 4))
    bars = plt.bar(labels, vals)
    plt.title(f'Comparison: {metric}')
    plt.ylim(0, max(1.0, max(vals) * 1.2))
    for bar, val in zip(bars, vals):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.3f}', ha='center', va='bottom')
    out = FIG_DIR / f'comparison_{metric}.png'
    plt.tight_layout()
    plt.savefig(out, dpi=150)
    plt.show()
    print('Saved figure:', out)
else:
    print(f'No values filled yet for {metric}.')

### Qualitative analysis prompts (for final write-up)

Use 3--5 examples and answer:
1. What issue type was solved/failed?
2. Did retrieval include the right file/function?
3. Was the output format valid SEARCH/REPLACE?
4. If failure: format error, retrieval miss, or patch logic error?
5. What change might fix that failure mode?

In [ ]:
check_paths = [
    'data/raw/raw_prs.jsonl',
    'data/raw/filtered_prs.jsonl',
    'data/processed/train.jsonl',
    'data/processed/val.jsonl',
    'data/processed/sft_cot_data.jsonl',
    'data/rag/faiss.index',
    'data/rag/chunks.jsonl',
    'outputs/grpo_student/final',
    'outputs/eval_student/raw_outputs.jsonl',
    'outputs/eval_student/all_preds.jsonl',
]

for p in check_paths:
    pp = ROOT / p
    print(f"{'OK  ' if pp.exists() else 'MISS'}  {p}")